# Multi asset and portfolio validation

This notebook evaluates **structured and regression models separately**, then compares matched cases. The portfolio contains **AAPL, AMGN, AMZN, AXP, BA, CAT, CRM, CSCO, CVX, DIS, GOOGL, GS, HD, HON, IBM, JNJ, JPM, KO, MCD, MMM, MRK, MSFT, NKE, NVDA, PG, SHW, TRV, UNH, V, WMT**, each with an initial weight of **1/30**, held without rebalancing. **AAPL is included. The company set matches the security dashboard; this is an equal-weight portfolio of the supplied companies, not an official index replication.**

Run the notebook from top to bottom. Existing simulation exports are read-only. Numerical identities can pass or fail; historical comparisons are diagnostic, with no invented model acceptance threshold. Defaults remain in portfolio losses. A missing holding never causes its weight to be reassigned.

The joint scenario construction matches the original row number across companies. Shared generator seeds can impose strong cross-company dependence; this notebook measures that existing design rather than fitting a new dependence model. Hypothetical scenarios compared with observed prices remain descriptive.

The final cell exports dashboard-ready JSON, Parquet and CSV tables plus separate model manifests. Use `execute_notebook.py` to additionally save a complete executed notebook and finalize the execution receipt **after all cells finish**.

In [ ]:
from pathlib import Path
import os, sys, json

PROJECT_ROOT = Path("/Users/gezimzeneli/Documents/ChatGPT/MScThesisV2")
IMPLEMENTATION_ROOT = PROJECT_ROOT / "porfolio_validation"
module_root = next((p for p in [Path.cwd(), IMPLEMENTATION_ROOT, Path.cwd() / "porfolio_validation"]
                    if (p / "portfolio_validation").is_dir()), None)
if module_root is None:
    raise RuntimeError("Keep this notebook with its portfolio_validation helper package.")
sys.path.insert(0, str(module_root))
from portfolio_validation.config import default_config, validate_config
from portfolio_validation.engine import PortfolioValidationSession
import numpy as np
import pandas as pd
from IPython.display import display

# Normal Run All processes the complete requested grid. The runner can override
# these settings through a recorded configuration file without editing the master.
CONFIG = default_config(PROJECT_ROOT, IMPLEMENTATION_ROOT / "results")
CONFIG.update(
    years=[2024, 2025, 2026],
    models=["structured", "regression"],
    scenarios=["default", "zero_shock", "oil_shock", "interest_rate_shock"],
    temporal_horizons=[1, 5, 20],
    scenario_horizons=[1, 5, 20, 63, 126, "terminal"],
    confidence_levels=[0.95, 0.99],
    bootstrap_replications=1000,
    bootstrap_block_lengths=[20, 10, 40],
    as_of="2026-11-13",
    save_path_details=True,
    save_portfolio_paths=False,  # Full wealth paths can be large; metrics/tail IDs are retained.
)
if os.environ.get("PORTFOLIO_VALIDATION_CONFIG"):
    CONFIG.update(json.loads(Path(os.environ["PORTFOLIO_VALIDATION_CONFIG"]).read_text()))
CONFIG = validate_config(CONFIG)
session = PortfolioValidationSession(CONFIG)
pd.set_option("display.max_columns", 16)
pd.set_option("display.max_rows", 16)

def preview(table, model=None, limit=12):
    frame = session.table(table, model)
    print(f"{table}: {len(frame):,} records" + (f" — {model}" if model else ""))
    if frame.empty:
        print("No records for this selection; consult availability and issues.")
    else:
        display(frame.head(limit))
    return frame

print("Model pipelines:", CONFIG["models"])
print("Years:", CONFIG["years"], "Scenarios:", CONFIG["scenarios"])
print("Profile:", CONFIG.get("run_profile", "interactive_full"))
print("Output:", session.output_dir)
display(session.table("asset_universe"))

## Input inventory and provenance

The full default configuration contains 132 model/year/scenario slots: 129 supported cases and three documented regression oil −100% cases. Each supported portfolio case needs the same eleven companies and 10,000 rows. Original and shock-expansion manifests are read; staging copies and optional controls are excluded.

Validation checks exact date mappings, consecutive price columns, source/output hashes, initial prices and zero/default behavior. Missing, unsupported and invalid cases remain visible. Refreshed market history and the pinned generation source are recorded separately.

In [ ]:
session.prepare()
planned = session.table("planned_cases")
display(planned.groupby(["year", "model"]).size().rename("planned_cases").reset_index())

## Structured model implementation

This pipeline loads only structured exports. It retains the structural model's financial default flags and zero-price outcomes. Its dependence diagnostics, portfolio construction, tail risk and drawdowns are calculated independently of the regression pipeline.

The calculation runs once per selected case; later sections preview these cached result tables. Historical references are cached per year, and scenarios are processed in batches to limit memory use.

In [ ]:
if "structured" in CONFIG["models"]:
    session.run_model("structured")
    preview("run_inventory", "structured")
    preview("portfolio_risk", "structured")
else:
    print("Structured model was not selected in this execution configuration.")

## Regression model implementation

This pipeline loads only regression exports and preserves their actual default behavior. It never imports structural default classifications. Regression oil −100% is unsupported because the generator's logarithmic input is undefined; the case receives no assumed prices or risk values.

The same mathematical validation definitions are applied to this independent ensemble, making subsequent matched-case comparisons interpretable.

In [ ]:
if "regression" in CONFIG["models"]:
    session.run_model("regression")
    preview("run_inventory", "regression")
    preview("portfolio_risk", "regression")
else:
    print("Regression model was not selected in this execution configuration.")

## Joint scenario construction and eligibility

Portfolio scenario `s` combines original row `s` for the eleven companies. This is an explicit **as-exported same-row** assumption, not proof of historically calibrated dependence or matched primitive innovations between the two models.

Within-path temporal diagnostics use joint paths whose prices remain positive and finite for all companies. Across-scenario cumulative simple returns retain zero endpoints as −100% returns. If no complete positive joint paths survive, temporal statistics are unavailable while portfolio wealth and tail-risk calculations remain valid. Undefined correlations are null, not zero.

In [ ]:
preview("joint_scenario_design")
preview("default_summary")
cohorts = preview("eligibility")
if not cohorts.empty and "status" in cohorts:
    display(cohorts.loc[cohorts.status.astype(str).str.startswith("unassessed")].head(20))

## Historical reference and uncertainty

Historical prices are aligned to NYSE sessions before returns are computed. Gaps remain missing; an interval crossing a gap is not treated as a valid shorter return. Each year's reference ends before the simulation origin. Company/horizon thresholds stay fixed across model and shock comparisons.

Synchronized circular block resampling uses the same date rows for all companies. Pointwise historical ranges are provided for Pearson correlations and joint-event frequencies, with block-length sensitivity. **Spearman estimates are computed but their bootstrap intervals are not computed.** Variation across simulated paths is reported separately and is not labeled a confidence interval.

In [ ]:
preview("historical_reference")
preview("historical_thresholds")
preview("bootstrap_sensitivity")

## Cross asset correlations and diversification

Pearson and Spearman signed-return and absolute-return correlations are assessed for all 435 company pairs. Within-path log-return correlations are computed over dates separately inside each simulated path. Across-scenario correlations use cumulative simple returns at a fixed horizon. These axes are never pooled.

Reports include historical and simulated pair correlations, simulated-minus-historical gaps, the maximum absolute gap and RMSE over **defined pairs only**. Smaller discrepancies indicate closer historical resemblance, not a universal pass. Comparisons of historical temporal behavior with shock-conditioned across-scenario behavior remain descriptive.

In [ ]:
preview("correlation_errors", limit=24)
preview("correlations", limit=8)

# One representative heatmap per model. All cases remain available in the export.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image
from io import BytesIO

def show_figure(fig):
    data = BytesIO()
    fig.savefig(data, format="png", dpi=120, bbox_inches="tight")
    display(Image(data=data.getvalue()))
    plt.close(fig)

corr = session.table("correlations")
for model in CONFIG["models"]:
    if corr.empty:
        break
    selected = corr.loc[(corr.model == model) & (corr.dependence_axis == "within_path")
                        & (corr.horizon_sessions == 1) & (corr.measure == "pearson")
                        & (corr["transform"] == "signed")]
    if selected.empty:
        continue
    example_id = selected.case_id.iloc[0]
    selected = selected.loc[selected.case_id == example_id]
    matrix = np.full((len(CONFIG["tickers"]), len(CONFIG["tickers"])), np.nan)
    index = {ticker: i for i, ticker in enumerate(CONFIG["tickers"])}
    for row in selected.itertuples():
        matrix[index[row.ticker_i], index[row.ticker_j]] = row.difference
        matrix[index[row.ticker_j], index[row.ticker_i]] = row.difference
    np.fill_diagonal(matrix, 0.0)
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(np.ma.masked_invalid(matrix), cmap="RdBu_r", vmin=-2, vmax=2)
    ax.set(xticks=range(len(CONFIG["tickers"])), yticks=range(len(CONFIG["tickers"])), xticklabels=CONFIG["tickers"], yticklabels=CONFIG["tickers"],
           title=f"{example_id}\nPearson correlation gaps: simulated − historical")
    plt.setp(ax.get_xticklabels(), rotation=60, ha="right")
    fig.colorbar(im, ax=ax, label="Correlation difference")
    show_figure(fig)

## Simultaneous losses and extreme movements

Joint downside events use simple returns strictly below each company's fixed historical fifth percentile. Joint extreme movements use absolute **daily log returns** strictly above the historical 95th percentile, regardless of direction. The fifth percentile defines an event, not an acceptance threshold.

Each record contains both thresholds, event counts, denominators and frequencies. Within-path frequencies are calculated per path then averaged. At a fixed future date, absolute-movement events use that date's daily return, whereas downside events use the stated cumulative horizon; `evaluation_session` distinguishes the two. Defaults stay in across-scenario downside outcomes.

In [ ]:
events = preview("joint_events", limit=16)
if not events.empty:
    display(events.groupby(["model", "dependence_axis", "metric", "status"], dropna=False).size().rename("pair_records").reset_index())

## Portfolio construction and distributions

Every holding starts with weight $1/30$. With initial capital one, holdings are $q_i=(1/30)/P_{i,0}$ and remain fixed:

$$V_{s,t}=\sum_i q_iP_{s,i,t}=\frac{1}{30}\sum_i\frac{P_{s,i,t}}{P_{i,0}}.$$

An independent holdings-based reconstruction checks the normalized-price implementation. Terminal loss is $1-V_{s,T}$. A constituent reaching zero retains zero value; its allocation is not redistributed. The fan charts summarize the complete ensemble, including defaults.

In [ ]:
checks = preview("portfolio_integrity", limit=14)
if not checks.empty:
    display(checks.groupby(["model", "status"]).size().rename("checks").reset_index())
    display(checks.loc[checks.status == "fail"])

daily = session.table("portfolio_daily_summaries")
for model in CONFIG["models"]:
    if daily.empty:
        break
    selected = daily.loc[daily.model.eq(model)]
    if selected.empty:
        continue
    example_id = selected.case_id.iloc[0]
    selected = selected.loc[selected.case_id.eq(example_id)].sort_values("day")
    x = pd.to_datetime(selected.date)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.fill_between(x, selected.q05.to_numpy(float), selected.q95.to_numpy(float), alpha=.2, label="5–95% scenarios")
    ax.plot(x, selected.q50, label="Median")
    valid_observed = selected.observed_comparable.fillna(False).astype(bool)
    ax.plot(x[valid_observed], selected.loc[valid_observed, "observed_value"], color="black", label="Observed, matching dates")
    ax.set(title=example_id, ylabel="Portfolio value / initial capital")
    ax.legend()
    show_figure(fig)

## Portfolio tail risk and company contributions

VaR is the linearly interpolated empirical loss quantile. Expected shortfall averages exactly $m=\lceil(1-\alpha)N\rceil$ worst losses, using decimal-safe tail counts and deterministic scenario-ID ordering for ties. Every company contribution uses the same selected portfolio scenarios.

$$c_i=-\frac{w_i}{m}\sum_{s\in A_\alpha}R_{s,i},\qquad e_\alpha=\left|\sum_i c_i-ES_\alpha\right|.$$

Negative VaR/ES values represent gains and remain unchanged. Positive contributions increase portfolio loss; negative contributions offset it. Values are fractions of initial capital. A smaller simulated ES is not evidence of a more accurate model.

In [ ]:
preview("portfolio_risk", limit=28)
preview("es_reconciliation", limit=16)
contributions = session.table("es_contributions")
for model in CONFIG["models"]:
    if contributions.empty:
        break
    selected = contributions.loc[contributions.model.eq(model) & contributions.confidence_level.eq(CONFIG["confidence_levels"][0])]
    if selected.empty:
        continue
    example_id = selected.case_id.iloc[0]
    selected = selected.loc[selected.case_id.eq(example_id)].set_index("ticker").reindex(CONFIG["tickers"])
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(selected.index, selected.value * 100, color=np.where(selected.value >= 0, "#bd6b37", "#29857e"))
    ax.axhline(0, color="black", linewidth=.7)
    ax.set(title=f"{example_id} — ES contributions at {CONFIG['confidence_levels'][0]:.0%}",
           ylabel="Percentage points of initial capital")
    show_figure(fig)

## Maximum drawdown

For each complete portfolio path, including its initial value:

$$MDD_s=\max_t\left(1-\frac{V_{s,t}}{\max_{u\leq t}V_{s,u}}\right).$$

Drawdown must lie between zero and one for nonnegative portfolio wealth. Observed comparisons use identical completed dates and the same initial allocation. Missing historical dates are not interpolated. Full-year 2026 observed drawdown remains unavailable until all required dates have observations.

In [ ]:
preview("portfolio_drawdowns", limit=24)

## Matched model and scenario comparisons

Only complete cases with the same year, scenario, magnitude, horizon and metric are compared. Model differences are **structured minus regression**. Scenario differences are relative to each model/year's supplied zero-shock reference. These are aggregate distribution comparisons, not claimed path-by-path causal effects.

The two models retain their own default behavior and source revisions. A shared seed or row number is not proof of matched primitive draws across models. Unsupported endpoints remain visible, including regression oil −100%.

In [ ]:
session.compare()
preview("scenario_coverage", limit=18)
preview("model_comparisons", limit=24)
preview("scenario_comparisons", limit=20)

## Export for the HTML overview

The export has a combined `dashboard_manifest.json` and separate `structured` and `regression` sub-bundles. Summary datasets are strict JSON row arrays, Parquet and CSV. Large per-scenario metrics and tail membership remain in Parquet. Full wealth-path export is optional.

Every record retains its model, year, scenario, magnitude, horizon, cohort and status. The manifest records units, hashes, schemas, configuration and interpretation. `null` means unavailable, never zero. Use the year-specific status ledger for a selected year; counts are result records, not independent observed market cases.

This final cell writes the analytical bundle. The external runner saves the executed notebook after this cell returns, verifies every declared artifact and finalizes `notebook_execution.json`. Interactive notebook execution leaves the manifest as `analytical_exported` until that external finalization occurs.

In [ ]:
result_folder = session.export()
manifest = json.loads((result_folder / "dashboard_manifest.json").read_text())
display(pd.DataFrame([{"dataset": d["name"], "rows": d["rows"], "JSON": d["files"].get("json"),
                      "Parquet": d["files"].get("parquet")} for d in manifest["datasets"]]))
print("Combined dashboard:", result_folder / "dashboard_manifest.json")
for model, relative in manifest["model_exports"].items():
    print(model, "dashboard:", result_folder / relative)
preview("test_status_by_year", limit=24)
preview("issues", limit=20)
print("Execution completion is separate from mechanical pass/fail and statistical interpretation.")